In [1]:
# Install required libraries with CUDA support
!pip install -q torch

In [2]:
import torch
from llama_index.llms.ollama import Ollama

In [3]:

!ollama pull mistral
# Install llama-cpp-python with CUDA 12.x support
!pip install llama-cpp-python llama-index-llms-ollama

]11;?\pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ pulling manifest ⠴ pulling manifest 
pulling f5074b1221da: 100% ▕██████████████████▏ 4.4 GB                         
pulling 43070e2d4e53: 100% ▕██████████████████▏  11 KB                         
pulling 1ff5b64b61b9: 100% ▕██████████████████▏  799 B                         
pulling ed11eda7790d: 100% ▕██████████████████▏   30 B                         
pulling 1064e17101bd: 100% ▕██████████████████▏  487 B                         
verifying sha256 digest 
writing manifest 
success 


In [4]:
!pip install llama-index

In [5]:
from llama_cpp import Llama
import os
llm = Ollama(model="mistral", request_timeout=260.0)
#llm = Ollama(model="hf.co/mlx-community/Mistral-7B-Instruct-v0.3-4bit", request_timeout=260.0)

In [6]:
# Test with a RAG query
prompt = "What is RAG in the context of large language models?"
print(f"\nSending prompt: {prompt}")

response = llm.complete(prompt)

print("\nResponse:")
print(response.text)


Sending prompt: What is RAG in the context of large language models?

Response:
 In the context of large language models, RAG (Roughly Automated Grading) is a tool developed by Mistral AI. RAG is used for automatically evaluating and scoring the quality and coherence of the text generated by the language model. It helps in assessing the model's responses based on a set of predefined criteria, such as grammar, coherence, relevance, and appropriateness, making it a useful tool for model development and quality assurance.

RAG operates on a scale from 0 to 6, where 0 indicates the response is completely inappropriate, and 6 indicates the response is excellent. This provides a quantitative way to assess the performance of the language model and track its improvements over time during the training process.


## Integrating Open-Source LLMs into RAG (with a PDF!)

In [7]:
!pip install pymupdf
!pip install llama-index-llms-llama-cpp

In [8]:
!pip install llama-index-embeddings-huggingface

In [9]:
import fitz  # PyMuPDF

# Load the sample contract PDF
pdf_path = "sample_contr.pdf"
doc = fitz.open(pdf_path)

# Extract text from all pages
text = "\n".join([page.get_text() for page in doc])

print(f"Extracted {len(text.split())} words from the PDF.")

FileNotFoundError: no such file: '/content/sample_contract.pdf'

In [ ]:
from llama_index.core import VectorStoreIndex, Document, get_response_synthesizer
from llama_index.core.retrievers import VectorIndexRetriever
from llama_index.core.query_engine import RetrieverQueryEngine
from llama_index.core.settings import Settings
from llama_index.llms.llama_cpp import LlamaCPP
from llama_index.embeddings.huggingface import HuggingFaceEmbedding

# Configure the LLM
llm = LlamaCPP(
    model_path="/content/mistral.gguf",
    temperature=0.7,
    max_new_tokens=512,
    context_window=2048,
    model_kwargs={"n_gpu_layers": 1}
)

# Configure open-source embedding model
embed_model = HuggingFaceEmbedding(
    model_name="BAAI/bge-small-en-v1.5"  # Lightweight but effective embedding model
)

# Set as the default LLM and embedding model
Settings.llm = llm
Settings.embed_model = embed_model

# Create documents from your text
documents = [Document(text=text)]  # 'text' should be your document content

# Build index
index = VectorStoreIndex.from_documents(documents)

# Configure retriever
retriever = VectorIndexRetriever(
    index=index,
    similarity_top_k=2,  # Retrieve 2 most similar chunks
)

# Configure response synthesizer
response_synthesizer = get_response_synthesizer(
    response_mode="tree_summarize",
)


In [ ]:
query = "What are the late payment penalties in this contract?"

# Assemble query engine
query_engine = RetrieverQueryEngine(
    retriever=retriever,
    response_synthesizer=response_synthesizer,
)


# Test query
response = query_engine.query(query)
print(response)